# DocRank360 — Cross-Encoder + Bi-Encoder Ranking System

This notebook demonstrates the deployable project modules rather than
duplicating application logic inside notebook cells.

**Responsible use:** Do not use ranking outputs as the sole basis for hiring,
legal, immigration, compensation, or employment decisions. Do not process
private or confidential data in a public environment.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.settings import Settings
from src.dataset_loader import load_ranking_dataset
from src.ranking_engine import TwoStageRankingEngine


In [ ]:
settings = Settings.from_yaml(PROJECT_ROOT / "config.yaml")
dataset = load_ranking_dataset(
    settings.documents_path,
    settings.queries_path,
    settings.qrels_path,
)

print("Documents:", len(dataset.documents))
print("Queries:", len(dataset.queries))
print("Qrels:", len(dataset.qrels))
display(dataset.documents.head())


## Build or load the document index

The first run downloads `all-MiniLM-L6-v2` and encodes the small sample
corpus. Later runs can load the saved NumPy index.


In [ ]:
engine = TwoStageRankingEngine.from_settings(settings)
index_ms = engine.prepare_index()
print(f"Index preparation: {index_ms:.2f} ms")


## Stage 1 — Bi-encoder retrieval

In [ ]:
query = "How can I find similar quality complaints and corrective actions?"
retrieval = engine.retrieve(query, candidate_k=10)
display(
    retrieval.candidates[
        ["retrieval_rank", "document_id", "title", "bi_encoder_score"]
    ]
)


## Stage 2 — Cross-encoder reranking

In [ ]:
response = engine.search(query, candidate_k=10, rerank_k=5)
display(
    response.reranked_results[
        [
            "retrieval_rank",
            "reranked_rank",
            "rank_movement",
            "document_id",
            "title",
            "bi_encoder_score",
            "cross_encoder_score",
        ]
    ]
)
print(response.latency)


## Interpretation

The bi-encoder provides fast candidate coverage. The cross-encoder jointly
reads each query-document pair and can reorder the candidate set. Review
both improvements and regressions during manual error analysis.
